# 📄 Multi-Document PDF RAG Assistant

**Tech Stack:** pypdf · sentence-transformers · FAISS · TinyLlama-1.1B-Chat

Notebook ini menjalankan end-to-end RAG pipeline:
1. Install dependencies
2. Setup project structure
3. Download/upload PDF dataset
4. Indexing pipeline (PDF → chunks → embeddings → FAISS)
5. RAG query (retrieval + generation)
6. Evaluation
7. (Bonus) Launch Streamlit app via ngrok


## 0. Runtime Check
Pastikan GPU aktif: **Runtime → Change runtime type → T4 GPU**

In [ ]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')

## 1. Install Dependencies

In [ ]:
%%capture
!pip install pypdf==4.3.1 sentence-transformers==3.0.1 faiss-gpu==1.7.2 \
    transformers==4.44.0 accelerate==0.33.0 bitsandbytes==0.43.3 tqdm pandas
print('✅ Dependencies installed')

## 2. Clone Repository & Setup Structure

In [ ]:
import os

# ── Option A: clone from GitHub (replace with your repo URL after pushing) ──
# !git clone https://github.com/YOUR_USERNAME/nolimit-ds-test-raka.git
# %cd nolimit-ds-test-raka

# ── Option B: create structure locally (used for this notebook) ──
ROOT = '/content/rag_project'
for d in ['data/pdfs', 'src', 'app', 'outputs', 'faiss_index', 'flowchart']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

os.chdir(ROOT)
print(f'Working directory: {os.getcwd()}')

## 3. Upload Source Files

Jika tidak menggunakan `git clone`, upload semua file dari `src/` dan `app/` menggunakan cell di bawah, atau gunakan panel Files di sebelah kiri Colab.

In [ ]:
# Uncomment untuk upload manual
# from google.colab import files
# uploaded = files.upload()  # pilih semua .py dari src/ dan app/

## 4. Download PDF Dataset

Dua opsi: download dari Kaggle, atau upload PDF sendiri.

In [ ]:
# ── Option A: Kaggle dataset ──
# !pip install -q kaggle
# from google.colab import files
# files.upload()  # upload kaggle.json API key
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d DATASET_SLUG -p data/pdfs --unzip

# ── Option B: Upload PDF files manually ──
from google.colab import files
print('Upload your PDF files:')
uploaded = files.upload()
import shutil
for fname in uploaded:
    shutil.move(fname, f'data/pdfs/{fname}')
    print(f'  Moved → data/pdfs/{fname}')

## 5. Write Source Modules

Cell-cell berikut menulis ulang semua modul `src/` langsung ke disk Colab. Skip jika sudah upload via git clone.

In [ ]:
%%writefile src/loader.py
"""loader.py — Multi-PDF Loader"""
import os
from pathlib import Path
from typing import List, Dict, Any
from pypdf import PdfReader
from tqdm import tqdm

def load_pdfs(pdf_dir: str) -> List[Dict[str, Any]]:
    pdf_dir = Path(pdf_dir)
    if not pdf_dir.exists():
        raise FileNotFoundError(f'Directory not found: {pdf_dir}')
    pdf_files = sorted(pdf_dir.glob('*.pdf'))
    if not pdf_files:
        raise ValueError(f'No PDF files found in: {pdf_dir}')
    print(f'Found {len(pdf_files)} PDF file(s)')
    documents = []
    for pdf_path in tqdm(pdf_files, desc='Loading PDFs'):
        try:
            reader = PdfReader(str(pdf_path))
            for page_num, page in enumerate(reader.pages, start=1):
                text = page.extract_text()
                if text and text.strip():
                    documents.append({'filename': pdf_path.name,
                                      'page_number': page_num,
                                      'total_pages': len(reader.pages),
                                      'text': text.strip()})
        except Exception as e:
            print(f'  [WARN] {pdf_path.name}: {e}')
    print(f'Extracted {len(documents)} page(s)')
    return documents

def get_dataset_summary(pdf_dir: str) -> Dict[str, Any]:
    pdf_dir = Path(pdf_dir)
    pdf_files = sorted(pdf_dir.glob('*.pdf'))
    summary = {'total_files': len(pdf_files), 'files': []}
    total_pages = 0
    for pdf_path in pdf_files:
        size_kb = pdf_path.stat().st_size / 1024
        try:
            reader = PdfReader(str(pdf_path))
            pages = len(reader.pages)
            readable = True
        except Exception:
            pages = 0; readable = False
        total_pages += pages
        summary['files'].append({'filename': pdf_path.name,
                                  'size_kb': round(size_kb, 2),
                                  'pages': pages,
                                  'readable': readable})
    summary['total_pages'] = total_pages
    return summary

In [ ]:
%%writefile src/chunker.py
"""chunker.py — Text Chunking"""
from typing import List, Dict, Any

def chunk_documents(documents, chunk_size=500, chunk_overlap=100):
    if chunk_overlap >= chunk_size:
        raise ValueError('chunk_overlap must be < chunk_size')
    chunks = []
    chunk_id = 0
    for doc in documents:
        text, filename, page_number = doc['text'], doc['filename'], doc['page_number']
        start = 0; chunk_index = 0
        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end].strip()
            if chunk_text:
                chunks.append({'chunk_id': chunk_id, 'filename': filename,
                               'page_number': page_number, 'chunk_index': chunk_index,
                               'text': chunk_text, 'char_start': start,
                               'char_end': min(end, len(text))})
                chunk_id += 1; chunk_index += 1
            start += chunk_size - chunk_overlap
    print(f'Generated {len(chunks)} chunk(s) [size={chunk_size}, overlap={chunk_overlap}]')
    return chunks

In [ ]:
%%writefile src/embedder.py
"""embedder.py — Embedding Generation"""
import numpy as np
from sentence_transformers import SentenceTransformer

DEFAULT_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

class Embedder:
    def __init__(self, model_name=DEFAULT_MODEL, device='cpu'):
        print(f'Loading embedding model: {model_name}')
        self.model = SentenceTransformer(model_name, device=device)
        self.model_name = model_name
        self.device = device
        self.embedding_dim = self.model.get_sentence_embedding_dimension()
        print(f'Embedding dimension: {self.embedding_dim}')

    def encode_chunks(self, chunks, batch_size=64, show_progress=True):
        texts = [c['text'] for c in chunks]
        embeddings = self.model.encode(texts, batch_size=batch_size,
                                       show_progress_bar=show_progress,
                                       convert_to_numpy=True,
                                       normalize_embeddings=True)
        print(f'Generated {embeddings.shape[0]} embeddings dim={embeddings.shape[1]}')
        return embeddings.astype('float32')

    def encode_query(self, query):
        vec = self.model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
        return vec.astype('float32')

In [ ]:
%%writefile src/vectorstore.py
"""vectorstore.py — FAISS Vector Store"""
import pickle
from pathlib import Path
import faiss
import numpy as np

class VectorStore:
    def __init__(self, embedding_dim):
        self.embedding_dim = embedding_dim
        self.index = faiss.IndexFlatIP(embedding_dim)
        self.metadata = []

    def add(self, embeddings, chunks):
        assert embeddings.shape[0] == len(chunks)
        self.index.add(embeddings)
        self.metadata.extend(chunks)
        print(f'VectorStore: {self.index.ntotal} vector(s) indexed')

    def search(self, query_vec, top_k=5):
        top_k = min(top_k, self.index.ntotal)
        scores, indices = self.index.search(query_vec, top_k)
        return [(self.metadata[i], float(s)) for s, i in zip(scores[0], indices[0]) if i != -1]

    def save(self, save_dir):
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        faiss.write_index(self.index, str(save_dir / 'faiss_index.bin'))
        with open(save_dir / 'metadata.pkl', 'wb') as f:
            pickle.dump(self.metadata, f)
        print(f'VectorStore saved → {save_dir}')

    @classmethod
    def load(cls, save_dir, embedding_dim):
        save_dir = Path(save_dir)
        store = cls(embedding_dim)
        store.index = faiss.read_index(str(save_dir / 'faiss_index.bin'))
        with open(save_dir / 'metadata.pkl', 'rb') as f:
            store.metadata = pickle.load(f)
        print(f'VectorStore loaded ({store.index.ntotal} vectors)')
        return store

In [ ]:
%%writefile src/retriever.py
"""retriever.py — Semantic Retriever"""
import sys; sys.path.insert(0, 'src')
from embedder import Embedder
from vectorstore import VectorStore

class Retriever:
    def __init__(self, embedder, vector_store):
        self.embedder = embedder
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5):
        query_vec = self.embedder.encode_query(query)
        return self.vector_store.search(query_vec, top_k=top_k)

    def format_context(self, results):
        parts = []
        for i, (chunk, score) in enumerate(results, 1):
            header = f'[Source {i}] {chunk["filename"]} (page {chunk["page_number"]}, score={score:.3f})'
            parts.append(f'{header}\n{chunk["text"]}')
        return '\n\n---\n\n'.join(parts)

    def format_citations(self, results):
        return [{'source_num': i, 'filename': c['filename'],
                 'page_number': c['page_number'], 'chunk_id': c['chunk_id'],
                 'score': round(s, 4),
                 'snippet': c['text'][:300] + ('...' if len(c['text']) > 300 else '')}
                for i, (c, s) in enumerate(results, 1)]

## 6. Dataset Exploration (Phase 1 Task 2)

In [ ]:
import sys, json
sys.path.insert(0, 'src')
from loader import get_dataset_summary
import pandas as pd

summary = get_dataset_summary('data/pdfs')
print(f"Total PDFs  : {summary['total_files']}")
print(f"Total pages : {summary['total_pages']}")
print()
df = pd.DataFrame(summary['files'])
display(df)

## 7. Indexing Pipeline

In [ ]:
import torch, sys
sys.path.insert(0, 'src')

from loader import load_pdfs
from chunker import chunk_documents
from embedder import Embedder
from vectorstore import VectorStore
from retriever import Retriever

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Step 1: Load PDFs
documents = load_pdfs('data/pdfs')

# Step 2: Chunk
chunks = chunk_documents(documents, chunk_size=500, chunk_overlap=100)

# Step 3: Embed
embedder = Embedder(device=DEVICE)
embeddings = embedder.encode_chunks(chunks)

# Step 4: Build FAISS index
vector_store = VectorStore(embedder.embedding_dim)
vector_store.add(embeddings, chunks)
vector_store.save('faiss_index')

# Step 5: Build retriever
retriever = Retriever(embedder, vector_store)
print('\n✅ Indexing pipeline complete')

## 8. Load TinyLlama LLM

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

LLM_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading {LLM_MODEL} on {DEVICE}...')

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto' if DEVICE == 'cuda' else None,
)
if DEVICE == 'cpu':
    llm = llm.to('cpu')

generator = hf_pipeline(
    'text-generation',
    model=llm,
    tokenizer=tokenizer,
    device=0 if DEVICE == 'cuda' else -1,
)
print('✅ LLM loaded')

## 9. RAG Query Function

In [ ]:
FALLBACK = 'I could not find relevant information in the document collection.'

PROMPT_TEMPLATE = """<|system|>
You are a helpful assistant. Answer questions strictly using the provided context.
If the answer is not in the context, say: \"{fallback}\"
</s>
<|user|>
Context:
{context}

Question: {question}
</s>
<|assistant|>
"""

def rag_query(question, top_k=5):
    results  = retriever.retrieve(question, top_k=top_k)
    context  = retriever.format_context(results)
    citations = retriever.format_citations(results)

    prompt = PROMPT_TEMPLATE.format(
        fallback=FALLBACK, context=context, question=question
    )

    output = generator(
        prompt,
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

    full_text = output[0]['generated_text']
    answer    = full_text[len(prompt):].strip()

    return {
        'question': question,
        'answer':   answer or FALLBACK,
        'sources':  citations,
    }

print('✅ rag_query() ready')

## 10. Run a Test Query

In [ ]:
# ── Edit question as needed ──
QUESTION = 'What is the main topic discussed in the documents?'

result = rag_query(QUESTION, top_k=5)

print('=' * 60)
print(f'QUESTION: {result["question"]}')
print('=' * 60)
print(f'ANSWER:\n{result["answer"]}')
print()
print('SOURCES:')
for src in result['sources']:
    print(f"  [{src['source_num']}] {src['filename']} — page {src['page_number']} (score={src['score']})")
    print(f"      {src['snippet'][:150]}...")
    print()

## 11. Evaluation (Phase 8)

In [ ]:
import pandas as pd

# ── Customize these evaluation questions ──
EVAL_QUESTIONS = [
    'What is the main topic discussed in the first document?',
    'Summarize the key points of the document collection.',
    'What specific information is provided about any technical topic?',
    'What conclusions or recommendations are made?',
    'Are there any data, statistics, or numbers mentioned?',
]

rows = []
for q in EVAL_QUESTIONS:
    r = rag_query(q, top_k=5)
    top_src = r['sources'][0] if r['sources'] else {}
    rows.append({
        'question':          q,
        'answer_preview':    r['answer'][:200],
        'top_source':        top_src.get('filename', '-'),
        'top_page':          top_src.get('page_number', '-'),
        'top_score':         top_src.get('score', 0),
        'has_fallback':      r['answer'] == 'I could not find relevant information in the document collection.',
    })

df_eval = pd.DataFrame(rows)
display(df_eval)

# Save results
df_eval.to_markdown('outputs/evaluation_results.md', index=False)
print('\n✅ Evaluation saved → outputs/evaluation_results.md')

## 12. Save Sample Results

In [ ]:
import json, datetime

sample_result = rag_query(QUESTION, top_k=5)

md_content = f"""# Sample RAG Results

Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

---

## Question
{sample_result['question']}

## Answer
{sample_result['answer']}

## Sources
"""

for src in sample_result['sources']:
    md_content += f"""\n### [{src['source_num']}] {src['filename']} — Page {src['page_number']}
- **Similarity Score:** {src['score']}
- **Snippet:** {src['snippet']}
"""

with open('outputs/sample_results.md', 'w') as f:
    f.write(md_content)

print('✅ Sample results saved → outputs/sample_results.md')
print(md_content[:500])

## 13. (Bonus) Launch Streamlit App via ngrok

Untuk menjalankan UI chatbot di Colab. Butuh akun ngrok gratis.

In [ ]:
# %%capture
# !pip install pyngrok streamlit
# from pyngrok import ngrok

# # Masukkan authtoken dari https://dashboard.ngrok.com/get-started/your-authtoken
# NGROK_TOKEN = 'YOUR_NGROK_TOKEN'
# ngrok.set_auth_token(NGROK_TOKEN)

# # Jalankan Streamlit di background
# import subprocess, threading
# def run_streamlit():
#     subprocess.run(['streamlit', 'run', 'app/streamlit_app.py',
#                     '--server.port', '8501', '--server.headless', 'true'])
# t = threading.Thread(target=run_streamlit, daemon=True)
# t.start()

# import time; time.sleep(5)  # wait for Streamlit to start
# tunnel = ngrok.connect(8501)
# print(f'\n🌐 Streamlit URL: {tunnel.public_url}')
print('Uncomment the cells above and add your ngrok token to launch the UI.')